In [ ]:
# =======================================================
# COLAB A: CENTRAL TAILSCALE API, WEB HOST & DATABASE
# =======================================================
import os
import time
import json
import subprocess
from google.colab import drive

print("📁 Mounting Google Drive to persist Tailscale identity & Database...")
drive.mount('/content/drive')

DRIVE_STATE_DIR = "/content/drive/MyDrive/Colab_A_Tailscale_State"
os.makedirs(DRIVE_STATE_DIR, exist_ok=True)
DRIVE_STATE_FILE = os.path.join(DRIVE_STATE_DIR, "tailscaled.state")

print("⚙️ Installing Flask, Waitress & Tailscale...")
os.system("pip install flask waitress > /dev/null 2>&1")
os.system("sudo mkdir -p /usr/share/keyrings >/dev/null 2>&1")
os.system("curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg | sudo tee /usr/share/keyrings/tailscale-archive-keyring.gpg >/dev/null")
os.system("curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list | sudo tee /etc/apt/sources.list.d/tailscale.list >/dev/null")
os.system("sudo apt-get update -y >/dev/null 2>&1")
os.system("sudo apt-get install -y tailscale >/dev/null 2>&1")

print("🔄 Booting Tailscale...")
os.system("sudo pkill -9 tailscaled >/dev/null 2>&1")
os.system("sudo rm -f /var/run/tailscale/tailscaled.sock >/dev/null 2>&1")
TS_STATE_DIR = "/var/lib/tailscale"
os.system(f"sudo mkdir -p {TS_STATE_DIR}")

if os.path.exists(DRIVE_STATE_FILE):
    os.system(f"sudo cp {DRIVE_STATE_FILE} {TS_STATE_DIR}/tailscaled.state")

os.system("nohup sudo tailscaled --tun=userspace-networking > tailscaled.log 2>&1 &")
time.sleep(3)
!sudo tailscale up
os.system(f"sudo cp {TS_STATE_DIR}/tailscaled.state {DRIVE_STATE_FILE}")

try: ts_ip = subprocess.check_output(["tailscale", "ip", "-4"]).decode("utf-8").strip()
except: ts_ip = "UNKNOWN"

# --- THE FLASK WEB HOST AND API ---
api_code = f"""
import logging
import time
import json
import threading
import queue
from flask import Flask, request, jsonify, render_template_string
from waitress import serve

log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)
app = Flask(__name__)

DB_PATH = "{DRIVE_STATE_DIR}/master_database.jsonl"

# --- SERVER MEMORY / STATE ---
data_queue = queue.Queue()
task_queue = queue.Queue()
server_stats = {{
    "items_saved": 0,
    "active_workers": set()
}}

# Pre-populate the Task Queue with a few test pages
for i in range(1, 6):
    task_queue.put(f"http://books.toscrape.com/catalogue/page-{{i}}.html")

# --- BACKGROUND DATABASE WRITER ---
def database_writer():
    while True:
        payload = data_queue.get()
        try:
            with open(DB_PATH, "a", encoding="utf-8") as f:
                f.write(json.dumps(payload) + "\\n")
            server_stats["items_saved"] += 1
            server_stats["active_workers"].add(payload.get('scraper_node', 'Unknown'))
        except Exception as e:
            pass
        finally:
            data_queue.task_done()

threading.Thread(target=database_writer, daemon=True).start()

# ==========================================
# ENDPOINT 1: THE INTERACTIVE WEB DASHBOARD
# ==========================================
@app.route('/', methods=['GET', 'POST'])
def web_dashboard():
    # If a user submitted a new URL from the web UI, add it to the queue!
    if request.method == 'POST':
        new_url = request.form.get('new_url')
        if new_url:
            task_queue.put(new_url)
            print(f"👨‍💻 [USER INPUT] Added new URL to queue: {{new_url}}")

    # The HTML UI (Now with an interactive form!)
    html_template = '''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Colab Master Node</title>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background-color: #0d1117; color: #c9d1d9; padding: 40px; }}
            .card {{ background-color: #161b22; border: 1px solid #30363d; border-radius: 10px; padding: 20px; max-width: 600px; margin: 0 auto; box-shadow: 0 4px 12px rgba(0,0,0,0.5); }}
            h1 {{ color: #58a6ff; text-align: center; border-bottom: 1px solid #30363d; padding-bottom: 10px; }}
            .metric {{ font-size: 18px; margin: 15px 0; display: flex; justify-content: space-between; }}
            .value {{ font-weight: bold; color: #3fb950; }}
            .alert {{ color: #ff7b72; }}
            .form-group {{ margin-top: 30px; border-top: 1px solid #30363d; padding-top: 20px; display: flex; gap: 10px; }}
            input[type="text"] {{ flex-grow: 1; padding: 10px; border-radius: 5px; border: 1px solid #30363d; background: #010409; color: white; font-size: 16px; }}
            button {{ padding: 10px 20px; background-color: #238636; color: white; border: none; border-radius: 5px; font-size: 16px; cursor: pointer; font-weight: bold; }}
            button:hover {{ background-color: #2ea043; }}
        </style>
        <!-- Note: Removed auto-refresh so you don't get interrupted while typing -->
    </head>
    <body>
        <div class="card">
            <h1>🚀 Master Scraper Dashboard</h1>
            <div class="metric"><span>Database Items Saved:</span> <span class="value">{{{{ items }}}}</span></div>
            <div class="metric"><span>URLs Left in Queue:</span> <span class="value {{{{ 'alert' if tasks == 0 else '' }}}}">{{{{ tasks }}}}</span></div>
            <div class="metric"><span>Active Worker Nodes:</span> <span class="value">{{{{ workers }}}}</span></div>

            <form class="form-group" method="POST" action="/">
                <input type="text" name="new_url" placeholder="Paste URL to scrape..." required>
                <button type="submit">Add Task</button>
            </form>
        </div>
    </body>
    </html>
    '''
    return render_template_string(
        html_template,
        items=server_stats["items_saved"],
        tasks=task_queue.qsize(),
        workers=len(server_stats["active_workers"])
    )

# ==========================================
# ENDPOINT 2: TASK DISPATCHER API
# ==========================================
@app.route('/get_task', methods=['GET'])
def get_task():
    try:
        next_url = task_queue.get_nowait()
        return jsonify({{"status": "success", "url": next_url}}), 200
    except queue.Empty:
        return jsonify({{"status": "done", "message": "No URLs left in queue!"}}), 200

# ==========================================
# ENDPOINT 3: DATA RECEIVER API
# ==========================================
@app.route('/data', methods=['POST'])
def receive_data():
    try:
        data = request.json
        data_queue.put(data)
        print(f"📥 [DATA SAVED] : {{data.get('item_title', 'Unknown Title')}}")
        return jsonify({{"status": "success"}}), 200
    except Exception as e:
        return jsonify({{"status": "error", "message": str(e)}}), 400

if __name__ == '__main__':
    serve(app, host='0.0.0.0', port=5000, threads=16)
"""

with open("api_server.py", "w") as f:
    f.write(api_code)

print("\n🚀 Launching Web Host & Production API...")
os.system("pkill -9 -f api_server.py")
os.system("nohup env PYTHONUNBUFFERED=1 python3 api_server.py > api_server.log 2>&1 &")
time.sleep(2)

print("="*60)
print(f"🎯 YOUR CENTRAL TAILSCALE IP IS: {ts_ip}")
print("="*60)
print(f"🌐 OPEN THIS LINK ON YOUR MAC/PHONE: http://{ts_ip}:5000")
print("\n👀 Listening for workers... \n")
!tail -f api_server.log

📁 Mounting Google Drive to persist Tailscale identity & Database...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚙️ Installing Flask, Waitress & Tailscale...
🔄 Booting Tailscale...

🚀 Launching Web Host & Production API...
🎯 YOUR CENTRAL TAILSCALE IP IS: 100.96.27.123
🌐 OPEN THIS LINK ON YOUR MAC/PHONE: http://100.96.27.123:5000

👀 Listening for workers... 

📥 [DATA SAVED] : All products | Books to Scrape - Sandbox
📥 [DATA SAVED] : All products | Books to Scrape - Sandbox
📥 [DATA SAVED] : All products | Books to Scrape - Sandbox
📥 [DATA SAVED] : All products | Books to Scrape - Sandbox
📥 [DATA SAVED] : All products | Books to Scrape - Sandbox
👨‍💻 [USER INPUT] Added new URL to queue: https://books.toscrape.com/catalogue/page-50.html
👨‍💻 [USER INPUT] Added new URL to queue: https://books.toscrape.com/catalogue/page-50.html
📥 [DATA SAVED] : All products | Books to Scrape - Sandbox
📥 [DATA SAVED] : All prod